# tinyformer on a T4 (Kaggle / Colab)

The larger runs that do not fit the author's laptop CPU: a ~17M-parameter TinyStories LM (and its RoPE / GQA variants) and a longer CIFAR-10 ViT run.

**Status: this notebook has not been run yet; the README reports no numbers from it.** Every cell writes the same `log.jsonl` format as the CPU runs, so `scripts/ablate_lm.py report` style plots work on its output.

In [ ]:
!git clone https://github.com/vik-bit/tinyformer.git
%cd tinyformer
!pip install -q -e '.[ref]'
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

## Data: first ~500 MB of TinyStories V2 (GPT-4) train, CDLA-Sharing-1.0
Tokenizer (8192 vocab) is this repo's BPE, trained on the first 50 MB.

In [ ]:
!mkdir -p data/big
!curl -sL https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt | head -c 500000000 > data/big/ts_train.txt
!python scripts/prepare_corpus.py data/big/ts_train.txt --vocab 8192 --out data/big

## ~17M-parameter LM (d=384, 8 layers, 6 heads, seq 512), fp16 autocast

In [ ]:
COMMON = '--vocab 8192 --data_dir data/big --d_model 384 --n_layer 8 --n_head 6 --seq 512 --batch 64 --steps 10000 --lr 1e-3 --warmup 500 --eval_every 500 --eval_batches 50 --device cuda --amp'
!python scripts/train_lm.py {COMMON} --positions rope --out runs/t4/rope_mha

### Ablations at this size (each ~same budget)

In [ ]:
!python scripts/train_lm.py {COMMON} --positions learned --out runs/t4/learned_mha
!python scripts/train_lm.py {COMMON} --positions rope --n_kv_head 2 --out runs/t4/rope_gqa2
!python scripts/train_lm.py {COMMON} --positions rope --norm rms --mlp swiglu --out runs/t4/rope_rms_swiglu

### Samples and KV-cache speed on GPU

In [ ]:
import sys, torch; sys.path.insert(0, '.')
from tinyformer.model import GPT, Config
from tinyformer.tokenizer import Tokenizer
tok = Tokenizer.load('data/big/tok8192.json')
ck = torch.load('runs/t4/rope_mha/model.pt', map_location='cuda')
m = GPT(Config(**ck['config'])).cuda().eval(); m.load_state_dict(ck['model'])
x = torch.tensor([tok.encode('Once upon a time, a little dog')], device='cuda')
print(tok.decode(m.generate(x, 200, temperature=0.8, top_k=50)[0].tolist()))

## CIFAR-10 ViT, longer schedule

In [ ]:
!python scripts/train_vit_cifar.py --d 256 --layers 8 --heads 8 --epochs 100 --batch 256 --lr 1e-3 --device cuda --threads 2 --out runs/t4/vit_cifar